# Lab — RAG in a nutshell (student version)

Anthony Réveillac — INSA Toulouse

This lab follows the lecture *RAG in a nutshell*. It is organised in three parts:

1. **Building a RAG from scratch**: foundation model, embedding model, chunks and splitter, retriever, RAG architecture, first experiments.
2. **Quantitative LLM evaluation in a nutshell**: similarity-based indicators and a first notion of faithfulness.
3. **An introduction to multimodal LLM and multimodal retrieval**: CLIP embeddings and an ANNOY index.

**Folders needed next to this notebook**

- `Docs/`: `Robert_Redford.txt`, `Personalities.txt`
- `Docs2/`: `BHT.txt`, `Interstellar.txt`
- `Images/`: the image database used in Part 3
- `solutions/`: the solutions of the cells to complete

**Cells to complete**

Some cells contain `# TODO` comments: complete them. If you are stuck, each of these cells ends with a commented line `# %load solutions/...`: uncomment it and run the cell. The cell is then replaced by the solution; run it a second time to execute it.


# Imports

In [ ]:
# Imports 

# General imports
import numpy as np
import re
import pandas as pd
import matplotlib.pyplot as plt

# Pytorch and transformers (for LLM)
import transformers, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, AutoModel
transformers.logging.set_verbosity_info()

# For loading documents from a path
from pathlib import Path

# For the embedding module
from sentence_transformers import SentenceTransformer, CrossEncoder

# %%

# Load device

if torch.backends.mps.is_available():
    # MPS is the GPU model in Mac technology
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device =torch.device("cpu")

print (torch.ones(1, device=device))

# Part 1 — Building a RAG from scratch

## The different Modules of the RAG

### The foundation model

We are going to work with Foundation Models, that is models that have been pre-trained on large data sets; but we want to work with OpenSource and hosted models. To do so we are going to fetch models from the [HuggingFace platform](https://huggingface.co/)

In [ ]:
class FoundationModel():

    def __init__(self,FOUND_MODEL_PATH,TEMPERATURE=None,MAX_NEW_TOKENS=10000):
        
        self.model=AutoModelForCausalLM.from_pretrained(FOUND_MODEL_PATH, 
                                             #device_map=mps_device,
                                             #device_map=cuda,
                                             torch_dtype="auto",
                                             trust_remote_code=True,
                                             ).to(device)
        
        
        self.tokenizer= AutoTokenizer.from_pretrained(FOUND_MODEL_PATH)

        self.model.generation_config.temperature=TEMPERATURE # Config of the temperature
        self.model.generation_config.top_p=None              # Config parameter related to the type of generation (like greedy decoding for instance)

        self.llm = pipeline("text-generation",
                     model=self.model,
                     tokenizer=self.tokenizer,
                     return_full_text=False,
                     max_new_tokens=MAX_NEW_TOKENS,
                     do_sample=True
                     )
        
        self.num_parameters = self.model.num_parameters()
        
        print('Number of parameters in my model','{:.2e}'.format(self.num_parameters))


    def generate_response(self,prompt):
        
        messages = [
            {'role':'user', 'content':prompt}
            ]
        
        output=self.llm(messages)
        # Note that the output is a list of len 1 which is a dict with key 'generated_text'
        return output


    # We anticipate the use of RAG and create a generate response taking into account the context

    def generate_response_with_context(self, prompt, context):
    
        # The context is a list of str 
    
        messages = []

        if context:
            for i, ctx in enumerate(context):
                messages.append({'role': 'system','content': f"context {i+1}: {ctx}"})

        messages.append({'role': 'user', 'content': prompt})

    
        output=self.llm(messages)
        return output

As you will see output of the models will be a list on len 1 in dict format with key ```generated_text```. We somehow reformat the output.

In [ ]:
def extract_response(output):
    # output is a list of len 1 as a dict with key 'generated_text'

    return output[0]['generated_text']

Another component of the answer is the inclusion of a reasoning component (which is identified with tags ```<think> ...</think>```). We write which only extract the answer.

In [ ]:
def short_response(output):

    response=extract_response(output=output)
    #text = "Before <think>to delete</think> After"
    short = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL)
    return short.strip()

#### Unit test for the foundation model

In [ ]:
# Unit test
# Start with a model
# Here we list some models and choose a small model Qwen 0.6

Path_SDS="HuggingFaceTB/SmolLM3-3B"
Path_Qwen_4B = "Qwen/Qwen3-4B"
Path_DSR1="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
Path_Q_06="Qwen/Qwen3-0.6B"

f_model = FoundationModel(FOUND_MODEL_PATH=Path_Q_06)

Have a look at the different elements while loading the model; do you recognize some constitutive elements of Transformers ? Look also the number of parameters of the model.

Try several prompts, look at the type of the output directly from the llm or simply the output produced with `generate_response`methods (standard or short).

In [ ]:
# Launch a query

# print the output and look to the format

#### How much memory does the model need?

Recall from the lecture (*Reality principle* slide) that the size of a model is given by

`Model_Size = Number of parameters × Precision (in bytes per parameter)`

Compute the theoretical size of your foundation model and compare it with the memory actually used. What would be the size of `Qwen/Qwen3-4B`? Of a 7B model in float32? Keep in mind that during generation the KV cache and the activations come on top of it.

In [ ]:
# Memory footprint of the foundation model

# TODO: get the number of bytes used to store one parameter (look at f_model.model.dtype)
bytes_per_param = ...

# TODO: compute the theoretical size of the model (in bytes) and print it in GB
theoretical_size = ...

# TODO: compare with f_model.model.get_memory_footprint() and, if you have a GPU,
#       with the memory actually allocated (torch.cuda.memory_allocated() or torch.mps.current_allocated_memory())

# Stuck? Uncomment the line below and run the cell twice (the solution replaces this cell)
# %load solutions/vram.py

We are going to study a specific example. Generate and print the answer to the query below. Which natural remark once have to keep in mind while using Open Source models ? 

In [ ]:
# A specific prompt

prompt_RF='Is Robert Redford alive ?; Answer must be [Yes] or [No]'

In [ ]:
# TODO: generate the answer of f_model to prompt_RF and print it (use short_response to remove the <think> part)

# Stuck? Uncomment the line below and run the cell twice (the solution replaces this cell)
# %load solutions/specific_query.py

### The Embedding model

Our foundation model will be mostly used as a decoder, even though it includes an encoder one usually chooses a separate embedding model. Indeed the embedding model of the decoder is designed for the generation process but fails to produce most adequate embeddings for a semantic study (like computing similarity). We make use here of an Open Source embedding model from Library `SentenceTransformer` (in particular we will create embeddings for sentences or more precisely for *chunks*).

In [ ]:
class EmbeddingModel():

    def __init__(self,EMBEDD_MODEL_PATH):

        # EMBEDD_MODEL_PATH is the name of the embedding model used within the SentenceTransformer lib

        # TODO: load the SentenceTransformer model on the device (self.Embedmodel)
        #       and store the dimension of its embeddings (self.dim)
        pass


    def get_embeddings(self,texts):

        # texts is a list of strings (which is supposed to be the list of chunks; without the source)
        # TODO: return normalized embeddings of torch type with shape (len(texts),self.dim)
        pass


    def compute_cos_sim_embed(self,embed1,embed2):

        # embed1,embed2 are two embeddings of shape (1,dim)
        # TODO: return the cos-similarity of the two embeddings (as a float)
        pass


    def compute_cos_sim_texts(self,text_1,text_2):

        # text_1,text_2 are two str
        # TODO: return the cos-similarity of the two texts (as a float)
        pass

# Stuck? Uncomment the line below and run the cell twice (the solution replaces this cell)
# %load solutions/embedding_class.py

#### Unit test for the embedding model

In [ ]:
# Unit test

Embed_mini="all-MiniLM-L6-v2"
EmbedModel=EmbeddingModel(EMBEDD_MODEL_PATH=Embed_mini)

# Once again have a look at the parameters of the model.

sentences=['Hello World','How is the weather ?']

embeddings = EmbedModel.get_embeddings(texts=sentences)

print(type(embeddings),embeddings.shape,type(embeddings[0].dtype))

em = EmbedModel.compute_cos_sim_embed(embeddings[0],embeddings[1])

print(em)

### Notion of Chunk and Splitter

We decide to create a class for managing chunks. A *chunk* will be defined by:

-  ```source``` the name of the .txt file from which the chunk comes from
-  ```content``` which is the str that composes the chunk
-  ```embedding``` which is the embedding associated to ```content```.

In [ ]:
class Chunk():

    def __init__(self,source,content,embed_model: EmbeddingModel):

        self.embedding_model=embed_model

        #dim is the common dimension of the embeddings
        dim = self.embedding_model.dim
        
        # A chunk is defined by its source (str); its content (str); its embedding (a torch which shape (1,dim))

        self.source=str(source)
        self.content=str(content)
        self.embedding=self.embedding_model.get_embeddings(texts=[content]).reshape(1,dim)


    def print_chunk(self):

        print('source:',self.source,'content:',self.content,'embedding shape:',self.embedding.shape)

### Towards the index: the splitter

We are going to chunk resources and then compare the similarity of each of these chunks with the one of the query. We follow the plan of the lecture and start with the splitter.

This class aims in getting the documents that will be in a folder with address ```path_doc```, and return as an output of the method ```get_chunks``` the chunks associated to these resources. Note that we keep track of which document is issued each chunk (via the ```source``` feature).

In [ ]:
class Splitter():

    def __init__(self,embed_model: EmbeddingModel):

        self.embedding_model=embed_model

        self.docs = []
        # We store the original documents as a list of .txt files (format is {"source":'File_name',"content_page":(str)})
        self.chunks=[]
        # This will be the list of chunks

    def get_documents(self,path_doc):
        # PATH_DOC is the Path form where the documents will be found (each document is a.txt file).
        docs=[]

        for file in Path(path_doc).rglob("*.txt"):
            name=file.name
            with open(file, "r", encoding="utf-8") as file:
                resource=file.read().strip()
                if resource:
                    docs.append({"source":name,"content_page":resource})

        self.docs=docs


    def get_chunks_contents_from_1_doc(self,file_name,content_page,chunk_size,overlap,sentence_split=False):

        # TODO: split content_page into chunks and append them (as Chunk objects) to self.chunks
        #   - if sentence_split: one chunk per sentence
        #   - otherwise: chunks of chunk_size characters, two consecutive chunks sharing overlap characters
        #     (raise an exception if overlap > chunk_size)
        pass


    def get_chunks(self,path_doc,chunk_size,overlap,sentence_split=False):

        # TODO: load the documents of path_doc and chunk each of them
        pass

    def reset_splitter(self):

        self.docs=[]
        self.chunks=[]

# Stuck? Uncomment the line below and run the cell twice (the solution replaces this cell)
# %load solutions/splitter_class.py

#### Unit test for the splitting module

In [ ]:
# Unit test

this_path = Path.cwd()/"Docs"
embed_model=EmbeddingModel(EMBEDD_MODEL_PATH=Embed_mini)

Split=Splitter(embed_model)

Split.reset_splitter()
Split.get_documents(path_doc=this_path)
Split.get_chunks(path_doc=this_path,chunk_size=30,overlap=15,sentence_split=True)

print(len(Split.chunks))

In [ ]:
# Do your own tests, with overlaps, without, what is the impact ? like for example : 

for i in range(len(Split.chunks)):

    print(Split.chunks[i].source,"\n",Split.chunks[i].content,"\n",Split.chunks[i].embedding.shape)

### Index, Database and Retriever

We are now in position to define the retriever. In our setting with very few document we will build our index (when the number of resources is pretty high (over $10^4$) we make use of an index (which is trained for that purpose); one can for example make use of the FAISS library.

In [ ]:
class Retriever():

    def __init__(self,embed_model: EmbeddingModel):

        self.embedding_model=embed_model

        # The index is a list of (Id(int),chunk); chunk needs the size DIM for the Embeddings
        self.index=[]


    def add_elements_to_index(self,chunks):

        # chunks is a list of chunk
        # TODO: add each chunk to self.index with an Id (int)
        pass

    def search_best(self,query,number_of_hits=3,adapt=False):

        # query is a str
        # TODO: embed the query, compute its cos similarity with every chunk of the index and
        #       return the number_of_hits best results as a list of (Id, chunk, sim), sorted by decreasing sim
        # TODO (adapt=True): add a criterion to exclude the worst hits
        #       (for instance: stop when the similarity is smaller than half of the previous one)
        pass

    def reset_Retriever_index(self):

        self.index=[]

# Stuck? Uncomment the line below and run the cell twice (the solution replaces this cell)
# %load solutions/retriever_class.py

#### Unit test of the Retriever module

In [ ]:
# Unit test

# Recall that we have already in our previous unit tests defined an embedding model and a splitter

        # embed_model=EmbeddingModel(EMBEDD_MODEL_PATH=Embed_mini)

        # Split=Splitter(embed_model)
        # Split.chunks=[]
        # Split.get_chunks(path_doc=this_path,chunk_size=30,overlap=15,sentence_split=True)

print(Split.docs)

chunks=Split.chunks

print(len(chunks))

retriever = Retriever(embed_model)


# TODO: add the chunks to the index

# TODO: get the best results using your retriever to the query below (try with and without adapt)

query='Is Robert Redford alive'

# Stuck? Uncomment the line below and run the cell twice (the solution replaces this cell)
# %load solutions/retriever_specific_prompt.py

In [ ]:
# Unit test follow up

Split.chunks=[]

Split.get_chunks(path_doc=this_path,chunk_size=30,overlap=15,sentence_split=True)

print(Split.docs)

chunks=Split.chunks

print(len(chunks))

retriever = Retriever(embed_model)

retriever.add_elements_to_index(chunks=chunks)

query='Is Robert Redford alive'

results=retriever.search_best(query=query,number_of_hits=2)

print("results",results)


## The RAG Architecture

In [ ]:
class RAG():

    def __init__(self,CONFIG):
        
        self.foundation_model=FoundationModel(FOUND_MODEL_PATH=CONFIG['FOUND_MODEL_PATH'])
        self.Embedding_model=EmbeddingModel(EMBEDD_MODEL_PATH=CONFIG['EMBEDD_MODEL_PATH'])
        self.splitter=Splitter(self.Embedding_model)
        self.retriever=Retriever(self.Embedding_model)

        self.dim_embed = CONFIG['DIM_EMBED']
        self.chunk_size = CONFIG['CHUNK_SIZE']
        self.overlap = CONFIG['OVERLAP']

    
    def reset_index(self):

        self.retriever.reset_Retriever_index()
        self.splitter.reset_splitter()
    
    
    def load_documents_and_get_chunks(self,path,sentence_split=False):

        self.splitter.get_chunks(path_doc=path,
                                 chunk_size=self.chunk_size,
                                 overlap=self.overlap,
                                 sentence_split=sentence_split)
        
        chunks = self.splitter.chunks

        self.retriever.add_elements_to_index(chunks=chunks)
        
    
    def get_retrieval(self,query,number_of_hits,adapt=False):

        retrieved_info = self.retriever.search_best(query=query,number_of_hits=number_of_hits,adapt=adapt)

        # It is the full information of the form (Id, chunk, sim)

        retrieved=[]
        
        for elem in retrieved_info:

            i,chunk, distance=elem

            retrieved.append(chunk.content)
        
        # We get rid of repeated items
        return list(dict.fromkeys(retrieved))
    
    def generate_response_rag(self,query):

        retrieved=self.get_retrieval(query=query,
                                          number_of_hits=3)
        
        return self.foundation_model.generate_response_with_context(prompt=query,
                                                                   context=retrieved)


## Experiments

In [ ]:
CONFIG = {
    'FOUND_MODEL_PATH':Path_Q_06,
    #'FOUND_MODEL_PATH':"Qwen/Qwen3-4B",
    'EMBEDD_MODEL_PATH':"all-MiniLM-L6-v2",
    'DIM_EMBED':384,
    'CHUNK_SIZE':300,
    'OVERLAP':30
        }

rag = RAG(CONFIG=CONFIG)

In [ ]:
# We initialize our RAG with our documents

rag.reset_index()

rag.load_documents_and_get_chunks(path=Path.cwd()/"Docs",sentence_split=True)

chunks=rag.splitter.chunks

### The Robert Redford experiment

In [ ]:
# Once again for the prompt below compare the answer with and without RAG
        # prompt_RF='Is Robert Redford alive ?; Answer must be [Yes] or [No]'

query = prompt_RF

# TODO: answer without RAG (foundation model only)

# TODO: print the retrieved context (3 hits)

# TODO: answer with RAG and compare the two answers

# Stuck? Uncomment the line below and run the cell twice (the solution replaces this cell)
# %load solutions/experiment_RF.py

#### Reranking

You noticed that since you asked for a fixed ```number_of_hits``` some of the context might be pointless. We aim in preventing this and selecting the retrieved context. We provide three ways for that. 

**1st we plot the similarity and try to find an accurate split out of it.**

In [ ]:
# In both methods we start with our retrieved information

rag.reset_index()

rag.load_documents_and_get_chunks(path=Path.cwd()/"Docs",sentence_split=True)

retrieved_info = rag.retriever.search_best(query=query,number_of_hits=3)
# It is (Id,chunk,sim) and it is ordered by sim


In [ ]:
# retrieved_info is a list of (Id, chunk, sim), ordered by decreasing sim

# TODO: build a DataFrame `data` with columns ["Id","chunk","sim"] (use chunk.content)
#       and a DataFrame `selected_data` keeping only the chunks whose similarity is close enough
#       to the previous one (for instance: not more than 10% below)

# Stuck? Uncomment the line below and run the cell twice (the solution replaces this cell)
# %load solutions/rerank_1.py

**2nd method : *LLM as a judge***

This time we give the query and the retrieved information to a LLM called *judge (as a) LLM* and ask it to select the more suitable chunks.

In [ ]:
# We define the judge (it is always better to take another LLM and to do the opposite of what we are doing now: usually one chooses the judge to be a more powerful model; here for VRAM reasons we select a weaker one).DS_Store

judge_llm = FoundationModel(FOUND_MODEL_PATH=Path_SDS)


In [ ]:
context_chunks = data["chunk"].tolist()

#print(context_chunks)

#prompt = print(f"Given [{query}] tell me among the elements of [{context_chunks}] which ones are the more relevant to [{query}]; give me the answer as a list of elements of [{context_chunks}]")

prompt=f""" You are a clever assistant. Given the query: "{query}"; here is a list of resources "{context_chunks}". For each of those say which is useful to answer the query. Give your result by only listing the useful results.
"""

print(judge_llm.generate_response(prompt=prompt))

**3rd method: reranking with a *cross-encoder***

As seen in the lecture, our retriever is a *bi-encoder*: the query and the chunks are embedded separately and then compared with the cos similarity. A *cross-encoder* reads the query and a chunk **together** and returns a relevance score. It is more precise but slower, so we use it only on the few chunks returned by the retriever: we retrieve more hits than needed, rerank them, and keep the best ones for the decoder.

Note that the cross-encoder scores are not cos similarities (they can be negative): only the ranking matters.

In [ ]:
# The cross-encoder (a small model trained to score (query, passage) pairs)

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=str(device))

# TODO 1: retrieve 5 hits for `query` with our retriever (bi-encoder)

# TODO 2: score each (query, chunk) pair with reranker.predict and sort the chunks by decreasing score;
#         display the cos similarity and the cross-encoder score side by side

# TODO 3: keep the 2 best chunks and pass them to the decoder together with the query

# Stuck? Uncomment the line below and run the cell twice (the solution replaces this cell)
# %load solutions/rerank_cross_encoder.py

Compare the ranking given by the cos similarity with the one given by the cross-encoder. Try again with some queries of the personalities experiment (Part 2).

# Part 2 — Quantitative LLM evaluation in a nutshell

We reuse the `rag` object built in Part 1 (same `CONFIG`), so the models are not loaded a second time.

## Q&A task; definition of `query`, `retrieved_context`, `expected_response` and `response` 

In [ ]:
# We work with the personalities file

queries = [
"Who introduced the theory of relativity?",
"Who was the first computer programmer?",
"What did Isaac Newton contribute to science?",
"Who won two Nobel Prizes for research on radioactivity?",
"What is the theory of evolution by natural selection?"
]

print(queries)


expected_responses = [
"Albert Einstein proposed the theory of relativity, which transformed our understanding of time, space and gravity.",
"Ada Lovelace is regarded as the first computer programmer for her work on Charles Babbage's early mechanical computer, the Analytical Engine.",
"Isaac Newton formulated the laws of motion and universal gravitation, laying the foundation for classical mechanics.",
"Marie Curie was a physicist and chemist who conducted pioneering research on radioactivity and won two Nobel Prizes.",
"Charles Darwin introduced the theory of evolution by natural selection in his book 'On the Origin of Species'."
]

print(expected_responses)

In [ ]:
# We initialize our RAG with our documents

rag.reset_index()

rag.load_documents_and_get_chunks(path=Path.cwd()/"Docs",sentence_split=True)

chunks=rag.splitter.chunks

In [ ]:
# We get the retrieved context; here it is important to let the number of hits to be chosen by the model

retrieved_contexts=[rag.get_retrieval(query=q,number_of_hits=3,adapt=True) for q in queries]

print(retrieved_contexts)

In [ ]:
responses = [short_response(rag.foundation_model.generate_response_with_context(prompt=query,context=context))
             for query, context in zip(queries, retrieved_contexts)]

print(responses)

In [ ]:
# Similarly to before, test the index and the impact of the RAG on the queries above based on the reference (contrary to the previous situation the Open Source model already has a sufficient knowledge on these questions).

So naturally the first notion of evaluation and the less objective is human evaluation. We would like to introduce more quantitative indicators.

## Similarity evaluation functions

In [ ]:
# We first can simply compute similarities between 2 of the 3 elements Q, ER, RC.

def similarity_resp_expect(response, expected_responses):
    # response is a single str
    # expected_responses is the list of all expected_responses
    return [rag.Embedding_model.compute_cos_sim_texts(response,expected_response) for expected_response in expected_responses]

def similarity_resp_context(response, contexts):
    # response is a single str
    # contexts is the list of all retrieved contexts (if there are several sentence retrieved for a given query then they are concatenated)
    return [rag.Embedding_model.compute_cos_sim_texts(response,"".join(context)) for context in contexts]

def similarity_resp_query(response, query):
    # response is a single str
    return rag.Embedding_model.compute_cos_sim_texts(response,query) 


def evaluate_sim(query,expected_responses,response,contexts):
    
    l=[-1]*len(expected_responses)
    l[0]=similarity_resp_query(response, query)

    return {
        "Similarity resp. expected_response": similarity_resp_expect(response, expected_responses),
        "Similarity resp. contexts": similarity_resp_context(response, contexts),
        "Similarity resp. query": l,
    }

In [ ]:
# We test for the first query for example

sim = evaluate_sim(query=queries[0],expected_responses=expected_responses,response=responses[0],contexts=retrieved_contexts)
print(sim)

dfsim=pd.DataFrame(sim)

print(dfsim)

In [ ]:
# One can go further and try to go beyond the similarity by defining indicators for the semantic precision
# There are several ways to do so; here we simply give one example which could be related to the so-called 'faithfulness' and which tries to answer the question: "How much the answer is built from the retrieved context?"

def faithfulness_precision(response, retrieved_context, threshold=0.5):
    # Precision = 1 if the response is supported by at least one chunk
    supported = any(rag.Embedding_model.compute_cos_sim_texts(response,chunk) >= threshold for chunk in retrieved_context)
    return 1.0 if supported else 0.0

def faithfulness_recall(response, retrieved_context, threshold=0.5):
    # Recall = proportion of chunks supporting the answer
    if len(retrieved_context) == 0:
        return 0.0
    covered = sum(rag.Embedding_model.compute_cos_sim_texts(response,chunk) >= threshold for chunk in retrieved_context)
    return covered / len(retrieved_context)

def f1_score(precision, recall):
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)


def evaluate_faithfulness(response,retrieved_context,threshold=0.5):

    precision=faithfulness_precision(response=response, retrieved_context=retrieved_context, threshold=threshold)
    recall=faithfulness_recall(response, retrieved_context=retrieved_context, threshold=threshold)

    return {
        "Faithfulness precision": precision,
        "Faithfulness recall": recall,
        "Faithfullness F1": f1_score(precision=precision, recall=recall),
    }

In [ ]:
# We test for the first query for example

print(retrieved_contexts[0],len(retrieved_contexts[0]))

faithfulness = evaluate_faithfulness(response=responses[0],retrieved_context=retrieved_contexts[0])
print(faithfulness)


Test these functions on the different queries above. Note that the queries and references are pretty different from each others; though finding the relevant `retrieved_context`might be challenging.

### To go further: the Brief History of Time and Interstellar Q&A RAG

To go further one may consider the task related to documents which are even closer from the semantic point of view and consider the files BHT.txt and Interstellar.txt in the Docs2 folder. These two files are long extracts of two Wikipedia pages: one dedicated to the Interstellar movie and the other one to the documentary on Stephen Hawking's life. 

In [ ]:
# We initialize our RAG with our documents

rag.reset_index()

rag.load_documents_and_get_chunks(path=Path.cwd()/"Docs2",sentence_split=True)

chunks=rag.splitter.chunks

In [ ]:
queries = ["How was the income for the Nolan's movie?",
"Who was the main character of the biographical documentary?",
"How is the science in both movies?"
]
expected_responses =["""Interstellar raised $677 million as a direct income and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014""",
"The documentary tells the story of the famous physicist Stephen Hawking",
"""The science is well documented in both movies. In Interstellar, Caltech theoretical physicist and 2017 Nobel laureate in Physics[4]
Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar. It has
also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics.Since its premiere,
Interstellar gained a cult following,[5] and now is regarded by many sci-fi experts as one of the best science-fiction films of all
time. For Hawking life it is based on his bestseller book and Morris has studied the history and philosophy of science at Princeton
and later Berkeley."""
]

In [ ]:
retrieved_contexts=[rag.get_retrieval(query=q,number_of_hits=3,adapt=True) for q in queries]

for r in retrieved_contexts:
    print(len(r))

responses = [short_response(rag.foundation_model.generate_response_with_context(prompt=query,context=context))
             for query, context in zip(queries, retrieved_contexts)]

for r in responses:
    print(r,"\n")

Inspired from the methods above, proceed to the evaluation:

- Human evaluation of the answers (one can inspect the complete answer)
- Discuss the chunking strategy, what about the context length ? the overlapping ? 

## Evaluation methods with llm as a judge

The previous example showed how the search of relevant chunks is difficult and how important it might be on the result. Other methods involving llm have been developed we simply mention them and refer for instance to the [RAGAS library](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/).

# Part 3 — An introduction to multimodal LLM and multimodal retrieval and index search

We use the CLIP model provided by the `sentence-transformers` library (model `clip-ViT-B-32`, i.e. OpenAI's CLIP ViT-B/32): no extra installation is needed for CLIP. The ANNOY index requires the `annoy` library.

## Imports

In [ ]:
import os
from PIL import Image
try:
    from solutions.annoy_fallback import AnnoyIndex
except ModuleNotFoundError:
    from RAG.solutions.annoy_fallback import AnnoyIndex

# SentenceTransformer has already been imported at the beginning of the notebook

### CLIP

#### Visit the CLIP webpage: [CLIP](https://github.com/openai/CLIP?tab=readme-ov-file)

#### Loading the model on a given database

In [ ]:
# We load the CLIP model (through sentence-transformers); it encodes both texts and images in the same vector space

model = SentenceTransformer("clip-ViT-B-32", device=str(device))

In [ ]:
# We are going to load and collect the images which are all in subfolders of a root folder 'root_dir' (choose the path where the images of the project are located)

root_dir = Path.cwd()/"Images"
print(root_dir)

image_paths = []
# collects all the paths of subdirectories

for root, dirs, files in os.walk(root_dir):
    print("ROOT:", root)
    print("FILES:", files)
    for file in files:
        if file.lower().endswith((".png", ".jpg", ".jpeg")):
            image_paths.append(os.path.join(root, file))


In [ ]:
# We load the images from root_dir

images = [Image.open(path).convert("RGB") for path in image_paths]
print(len(images))

In [ ]:
# We create embeddings for the images (by batches, normalized)

image_embeddings = model.encode(images, batch_size=64, convert_to_numpy=True,
                                normalize_embeddings=True, show_progress_bar=True)

print(type(image_embeddings),image_embeddings.shape)

Getting the embeddings

In [ ]:
# Encoding a text

def encode_text(text):
    # returns a normalized embedding (numpy array of shape (512,))
    return model.encode(text, convert_to_numpy=True, normalize_embeddings=True)

In [ ]:
# Text embedding
text='rugby'
text_emb=encode_text(text)

print(type(text_emb),text_emb.shape)

In [ ]:
# Encoding an image

def encode_image(image_path):
    # returns a normalized embedding (numpy array of shape (512,))
    image = Image.open(image_path).convert("RGB")
    return model.encode(image, convert_to_numpy=True, normalize_embeddings=True)

In [ ]:
# Image embedding
image_path=image_paths[10]
im_emb=encode_image(image_path=image_path)

print(type(im_emb),im_emb.shape)

##### Computing cos similarities

In [ ]:
def compute_cos_sim(embed1,embed2):

    # embed1,embeds2 are two embeddings; careful they are already normalized
    # We compute the cos-similarity of two texts (it is returned as a float)

    return np.dot(embed1,embed2.T).item()

In [ ]:
# Embeddings text-text

print(compute_cos_sim(encode_text('rugby'),encode_text('ball')))
print(compute_cos_sim(encode_text('cat'),encode_text('tiger')))
print(compute_cos_sim(encode_text('cat'),encode_text('book')))

In [ ]:
# Embeddings text-image

plt.figure(figsize=(15, 4))
for i in range(525,528):
    image=Image.open(image_paths[i])
    plt.subplot(1, 5, i-525+1)
    plt.imshow(image)
    plt.axis("off")
    print(compute_cos_sim(encode_text('money'),encode_image(image_paths[i])))

plt.show()


**So the similarity can be pretty low, this is usually the case with CLIP; more precise description will improve the result**

In [ ]:
# Embeddings text-image

plt.figure(figsize=(15, 4))
for i in range(525,528):
    image=Image.open(image_paths[i])
    plt.subplot(1, 5, i-525+1)
    plt.imshow(image)
    plt.axis("off")
    print(compute_cos_sim(encode_text('a couple with Love or Money'),encode_image(image_paths[i])))

plt.show()

In [ ]:
# Embeddings image-image

plt.figure(figsize=(15, 4))
for i in range(525,528):
    image=Image.open(image_paths[i])
    plt.subplot(1, 5, i-525+1)
    plt.imshow(image)
    plt.axis("off")
    print(compute_cos_sim(encode_image(image_paths[525]),encode_image(image_paths[i])))

plt.show()


### Index using ANNOY

#### ANNOY is a search library belonging to the category of trees search based on the values of the embeddings. Visit the webpage: [ANNOY](https://github.com/spotify/annoy) to get a feeling of the architecture of this index. 

In [ ]:
dim = image_embeddings.shape[1]
index = AnnoyIndex(dim, metric="angular")

for i, emb in enumerate(image_embeddings):
    index.add_item(i, emb)

# The index search is based on random trees method and one has to initialize the size of the index (20 trees in the example below)
index.build(20)  

In [ ]:
# Encoding a text

def encode_text(text):
    # returns a normalized embedding (numpy array of shape (512,))
    return model.encode(text, convert_to_numpy=True, normalize_embeddings=True)

### Findings best hits from a request

The advantage of a multimodal model is to allow one to make requests either as a text input or an image input (in our examples where the modes are text and image).

##### Finding images from text

Given a text description; we look for the neighboring images.

In [ ]:
query = "a ninja with a sabre"
query_emb = encode_text(query)

results = index.get_nns_by_vector(query_emb, 5)
print(results)

plt.figure(figsize=(15, 4))
for i, idx in enumerate(results):
    plt.subplot(1, 5, i+1)
    plt.imshow(Image.open(image_paths[idx]))
    plt.axis("off")
plt.show()


#### Finding images from an image

Given an image; we look for the neighboring images.

In [ ]:
image=Image.open(image_paths[10])
plt.imshow(image)

q_emb = encode_image(image_paths[10])

results = index.get_nns_by_vector(q_emb, 5)

plt.figure(figsize=(15, 4))
for i, idx in enumerate(results):
    plt.subplot(1, 5, i+1)
    plt.imshow(Image.open(image_paths[idx]))
    plt.axis("off")
plt.show()

Note that our approach here using CLIP leads to a more semantic understanding of 'close images' compared to other tools (like the one you practiced in the lab on recommendation systems) where close images where images which looked alike (same panel of colors for examples), here the images do not look alike but they are closed in the sense they are topic-related. 